# Library Imports
Print python / package info and import libraries

## Parameters

In [2]:
import sys
import os
import glob
import json
import datetime
import getpass
import gc

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import seaborn as sns

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, LSTM
from tensorflow.keras.optimizers import Adam

from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.decomposition import PCA
from sklearn.metrics import mean_absolute_error

# Optional: reproducibility
SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

print("Imports OK. Python:", sys.version.splitlines()[0])

learning_rate = 0.0005
setEpoch = 50
batch_size = 32

# explicit data folder and csv
data_dir = r"C:\Users\nnaji\OneDrive\Documents\GitHub\UGRA_LSTM_Solar-Prediction\File Versions py files"
csv_path = os.path.join(data_dir, 'BigData.csv')
assert os.path.isfile(csv_path), f"CSV not found: {csv_path}"
print('Loading dataset:', csv_path)


Imports OK. Python: 3.13.7 (tags/v3.13.7:bcee1c3, Aug 14 2025, 14:15:11) [MSC v.1944 64 bit (AMD64)]
Loading dataset: C:\Users\nnaji\OneDrive\Documents\GitHub\UGRA_LSTM_Solar-Prediction\File Versions py files\BigData.csv


## Load data

In [3]:
df = pd.read_csv(csv_path)
df.head()

,MONTH,DAY,HOUR,MINUTE,TEMPERATURE,WINDANGLE,WINDSPEED,ZENITH,CDHI,CDNI,DHI,DNI,CLOUD,POWER
0,1,1,0,0,10.1,261,4.7,170.95,0,0,0,0,7,0.0
1,1,1,0,30,10.3,261,4.8,168.57,0,0,0,0,6,0.0
2,1,1,1,0,10.4,260,5.0,163.66,0,0,0,0,7,0.0
3,1,1,1,30,10.3,259,4.9,157.84,0,0,0,0,6,0.0
4,1,1,2,0,10.2,258,4.9,151.69,0,0,0,0,6,0.0


## Exploratory plots (compact)
Time-series overview, heatmap and boxplots; remove or reduce these later if too many features.

In [4]:
# # quick overview: small figure set
# # These plots give a quick visual overview of the features and their correlations. Currently Commented out. 
# df.plot(subplots=True, figsize=(12, 8), title='Feature Time Series')
# plt.tight_layout()
# #plt.show()

# plt.figure(figsize=(8,6))
# sns.heatmap(df.corr(), annot=True, fmt='.2f', cmap='coolwarm')
# plt.title('Feature Correlation Heatmap')
# #plt.show()


## Preprocessing: separate target, reshape, normalize

In [ ]:
# Preprocessing: use only the four requested features (no PCA)
selected_features = ['CDHI', 'CDNI', 'DHI', 'DNI']
missing = [c for c in selected_features if c not in df.columns]
if missing:
    raise KeyError(f"Missing required columns in dataframe: {missing}")

# Build X and y
X = df[selected_features].astype(float).copy()
y = df['POWER'] if 'POWER' in df.columns else None

# Scale features (StandardScaler used here; switch to MinMaxScaler if preferred)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Use scaled features directly for sequence construction (no PCA)
solar_Xdata = X_scaled        # shape (n_samples, n_features)
solar_ydata = y.values.reshape(-1, 1) if y is not None else None

# If a run_config dict exists, update it so records reflect the feature set
try:
    run_config['features_used'] = selected_features
    run_config['n_features'] = len(selected_features)
    run_config['scaler'] = 'StandardScaler'
    run_config['use_pca'] = False
except NameError:
    pass

print("Using features:", selected_features)
print("Feature matrix shape:", solar_Xdata.shape)


## Create sequences for LSTM
Using 24 time steps as in your .py file

In [ ]:
# --- Feature groups for fast cross-validation ---
feature_groups = {
    "A": ["CDHI", "CDNI", "DHI", "DNI"],                       # irradiance
    "B": ["TEMPERATURE", "WINDSPEED", "WINDANGLE", "CLOUD"],   # weather
    "C": ["MONTH", "DAY", "HOUR", "ZENITH"],                   # temporal + geometry
    "D": ["CDHI", "TEMPERATURE", "ZENITH", "CLOUD"]            # mixed (optional)
}

# ---- FAST MODE SETTINGS ----
# For quick experiments use a small subset of months and fewer epochs:
FAST_MONTHS = [1, 6, 12]   # quick passes: Jan, Jun, Dec
FAST_EPOCHS = 10           # faster than setEpoch

# To run full evaluation set:
# FAST_MONTHS = list(range(1,13))
# FAST_EPOCHS = setEpoch

import time
from sklearn.preprocessing import StandardScaler
from tensorflow.keras import backend as K

# helper: build sequences across full dataset (aligns months to sequence end like your original code)
def make_full_sequences(X_all, y_all, months_array, seq_len):
    X_seq, y_seq, months_seq = [], [], []
    for i in range(seq_len, len(y_all)):
        X_seq.append(X_all[i-seq_len:i, :])
        y_seq.append(y_all[i, 0] if y_all.ndim == 2 else y_all[i])
        months_seq.append(months_array[i])
    return np.array(X_seq), np.array(y_seq), np.array(months_seq)

def build_model(input_shape, lr=0.0005):
    model = Sequential()
    model.add(LSTM(units=50, return_sequences=True, input_shape=input_shape))
    model.add(LSTM(units=50))
    model.add(Dense(units=1))
    model.compile(optimizer=Adam(learning_rate=lr), loss='mean_absolute_error')
    return model

def run_group_fast(group_name, feature_list, months_array, full_df, y_array, month_names,
                   seq_len=24, fast_months=FAST_MONTHS, fast_epochs=FAST_EPOCHS, batch_size=32,
                   verbose=False):
    print(f"\n===== Group {group_name}: {feature_list} =====")
    missing = [c for c in feature_list if c not in full_df.columns]
    if missing:
        raise KeyError(f"Missing columns for group {group_name}: {missing}")

    # build raw feature matrix and ensure float
    X_all_raw = full_df[feature_list].astype(float).values
    y_all = y_array.reshape(-1, 1) if y_array.ndim == 1 else y_array

    # create sequences aligned as in your original pipeline
    X_seq, y_seq, months_seq = make_full_sequences(X_all_raw, y_all, months_array, seq_len)
    if verbose:
        print("Sequences created:", X_seq.shape)

    month_maes = []
    for idx, test_month in enumerate(fast_months, start=1):
        print(f"\n  Fold {idx}/{len(fast_months)} – testing {month_names[test_month-1]}")
        t0 = time.time()

        test_mask = (months_seq == test_month)
        train_mask = ~test_mask

        X_train = X_seq[train_mask]
        y_train = y_seq[train_mask]
        X_test  = X_seq[test_mask]
        y_test  = y_seq[test_mask]

        if len(X_test) < 10:
            print(f"    SKIPPED (only {len(X_test)} test sequences)")
            month_maes.append(np.nan)
            continue

        # scale features per-fold to avoid leakage (fit on training sequences flattened)
        ns, sl, nf = X_train.shape
        scaler = StandardScaler()
        X_train_flat = X_train.reshape(-1, nf)
        scaler.fit(X_train_flat)
        X_train = scaler.transform(X_train_flat).reshape(ns, sl, nf)
        # transform test set
        X_test = scaler.transform(X_test.reshape(-1, nf)).reshape(X_test.shape)

        # build and train model
        K.clear_session()
        model = build_model(input_shape=(X_train.shape[1], X_train.shape[2]), lr=learning_rate)
        model.fit(X_train, y_train, epochs=fast_epochs, batch_size=batch_size, validation_split=0.1, verbose=0)

        # predict and score
        y_pred = model.predict(X_test, verbose=0).flatten()
        test_mae = mean_absolute_error(y_test.flatten(), y_pred)

        month_maes.append(float(test_mae))
        print(f"    MAE = {test_mae:.4f}  |  time = {time.time() - t0:.1f}s")

        # cleanup
        del model
        K.clear_session()
        gc.collect()

    avg_mae = float(np.nanmean(month_maes))
    print(f"\n>> Group {group_name} average MAE over FAST_MONTHS = {avg_mae:.4f}")
    return avg_mae, month_maes

# --- Run fast cross-validation across groups ---
# Ensure months array exists (uses original dataframe MONTH column)
if 'MONTH' not in df.columns:
    raise KeyError("DataFrame missing MONTH column required for month-based CV. Add MONTH or use chronological split.")

months_full = df['MONTH'].values
month_names = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']

group_results = []
for g_name, feats in feature_groups.items():
    avg_mae, month_maes = run_group_fast(
        group_name=g_name,
        feature_list=feats,
        months_array=months_full,
        full_df=df,
        y_array=solar_ydata.flatten() if solar_ydata is not None else df['POWER'].values,
        month_names=month_names,
        seq_len=24,
        fast_months=FAST_MONTHS,
        fast_epochs=FAST_EPOCHS,
        batch_size=batch_size
    )
    group_results.append({
        "Group": g_name,
        "Features": ", ".join(feats),
        "Avg_MAE": avg_mae
    })

results_df = pd.DataFrame(group_results).sort_values("Avg_MAE")
print("\n\nQuick comparison (lower MAE is better):")
display(results_df)

TypeError: run_group_fast() got an unexpected keyword argument 'y'